In [1]:
import os
import json
import math
import copy

import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader

In [2]:
with open("data_split.json", "r") as f:
    split_data = json.load(f)

train_subjects = split_data["train"]
val_subjects = split_data["validation"]
test_subjects = split_data["test"]

print("Train:", len(train_subjects))
print("Validation:", len(val_subjects))
print("Test:", len(test_subjects))

Train: 1000
Validation: 125
Test: 126


In [3]:
train_data = r"Data/ASNR-MICCAI-BraTS2023-GLI-Challenge-TrainingData"

In [4]:
def preprocess_t2f(image):
    if image.shape != (240, 240, 155):
        raise ValueError(
            f"Unexpected image shape: {image.shape}"
        )

    # Crop to 208 x 224 x 155
    image = image[16:224, 8:232, :]

    # Pad depth to 160
    image = np.pad(
        image,
        ((0, 0), (0, 0), (2, 3)),
        mode="constant",
        constant_values=0
    )

    # Exact original BraTS foreground mask
    foreground = image > 0

    if not np.any(foreground):
        raise ValueError(
            "No foreground voxels found"
        )

    upper = np.percentile(
        image[foreground],
        99.9
    )

    image = np.clip(
        image,
        0,
        upper
    )

    # [0,1]
    image = image / upper

    # [-1,1]
    image = (
        image * 2.0
        - 1.0
    )

    # True air background
    image[~foreground] = -1.0

    return (
        image.astype(np.float32),
        foreground.astype(np.float32)
    )

In [5]:
class BraTSDataset(Dataset):
    def __init__(
        self,
        subjects,
        data_dir
    ):
        self.subjects = subjects
        self.data_dir = data_dir

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):

        subject = self.subjects[idx]

        subject_path = os.path.join(
            self.data_dir,
            subject
        )

        files = os.listdir(
            subject_path
        )

        t2f_files = [
            f for f in files
            if "t2f" in f.lower()
        ]

        if len(t2f_files) == 0:
            raise FileNotFoundError(
                f"No T2f file found for {subject}"
            )

        image = nib.load(
            os.path.join(
                subject_path,
                t2f_files[0]
            )
        ).get_fdata()

        image, _ = (
            preprocess_t2f(image)
        )

        image = torch.from_numpy(
            image
        ).unsqueeze(0)
        return {
            "image": image,
            "subject": subject
        }

In [6]:
train_dataset = BraTSDataset(
    subjects=train_subjects,
    data_dir=train_data
)

print("Dataset size:", len(train_dataset))

Dataset size: 1000


In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print(
    "Number of training batches:",
    len(train_loader)
)

Number of training batches: 1000


In [8]:
timesteps = 250


def cosine_beta_schedule(
    timesteps,
    s=0.008
):
    steps = timesteps + 1

    x = torch.linspace(
        0,
        timesteps,
        steps,
        dtype=torch.float64
    )

    alpha_bar = torch.cos(
        (
            (x / timesteps + s)
            / (1 + s)
        )
        * math.pi
        * 0.5
    ) ** 2

    alpha_bar = (
        alpha_bar
        / alpha_bar[0]
    )

    betas = (
        1.0
        - (
            alpha_bar[1:]
            / alpha_bar[:-1]
        )
    )

    return torch.clamp(
        betas,
        min=1e-8,
        max=0.999
    ).float()


# Original cosine schedule
betas = cosine_beta_schedule(
    timesteps
)

# V8: force terminal SNR to exactly zero
alphas = (
    1.0 - betas
)

alphas_cumprod = torch.cumprod(
    alphas,
    dim=0
)

alphas_cumprod_prev = F.pad(
    alphas_cumprod[:-1],
    (1, 0),
    value=1.0
)


sqrt_alphas_cumprod = torch.sqrt(
    alphas_cumprod
)

sqrt_one_minus_alphas_cumprod = (
    torch.sqrt(
        1.0
        - alphas_cumprod
    )
)


posterior_variance = (
    betas
    * (
        1.0
        - alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)

posterior_variance = torch.clamp(
    posterior_variance,
    min=1e-20
)


posterior_mean_coef1 = (
    betas
    * torch.sqrt(
        alphas_cumprod_prev
    )
    / (
        1.0
        - alphas_cumprod
    )
)


posterior_mean_coef2 = (
    (
        1.0
        - alphas_cumprod_prev
    )
    * torch.sqrt(
        alphas
    )
    / (
        1.0
        - alphas_cumprod
    )
)


snr = (
    alphas_cumprod
    / torch.clamp(
        1.0
        - alphas_cumprod,
        min=1e-12
    )
)


print(
    "Beta range:",
    betas.min().item(),
    betas.max().item()
)

print(
    "Initial alpha_cumprod:",
    alphas_cumprod[0].item()
)

print(
    "Final alpha_cumprod:",
    alphas_cumprod[-1].item()
)

print(
    "Final SNR:",
    snr[-1].item()
)


Beta range: 0.0001942687958944589 0.9990000128746033
Initial alpha_cumprod: 0.999805748462677
Final alpha_cumprod: 3.885928379077086e-08
Final SNR: 3.885928734348454e-08


In [9]:
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(
        self,
        dim
    ):
        super().__init__()

        self.dim = dim

    def forward(self, t):

        device = t.device

        half_dim = (
            self.dim // 2
        )

        embedding_scale = (
            math.log(10000)
            / (half_dim - 1)
        )

        embeddings = torch.exp(
            torch.arange(
                half_dim,
                device=device
            )
            * -embedding_scale
        )

        embeddings = (
            t[:, None].float()
            * embeddings[None, :]
        )

        embeddings = torch.cat(
            (
                embeddings.sin(),
                embeddings.cos()
            ),
            dim=1
        )

        return embeddings

In [10]:
class ResBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim,
        dropout=0.0
    ):
        super().__init__()

        self.norm1 = nn.GroupNorm(
            num_groups=8,
            num_channels=in_channels
        )

        self.conv1 = nn.Conv3d(
            in_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        self.time_mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(
                time_dim,
                out_channels * 2
            )
        )

        # V5:
        # Start scale = 0 and shift = 0
        nn.init.zeros_(
            self.time_mlp[-1].weight
        )

        nn.init.zeros_(
            self.time_mlp[-1].bias
        )


        self.norm2 = nn.GroupNorm(
            num_groups=8,
            num_channels=out_channels
        )

        self.dropout = nn.Dropout(
            dropout
        )

        self.conv2 = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1
        )

        # Start residual branch near zero
        nn.init.zeros_(
            self.conv2.weight
        )

        nn.init.zeros_(
            self.conv2.bias
        )


        if in_channels != out_channels:
            self.residual = nn.Conv3d(
                in_channels,
                out_channels,
                kernel_size=1
            )
        else:
            self.residual = nn.Identity()


    def forward(
        self,
        x,
        t
    ):
        residual = self.residual(
            x
        )

        h = self.norm1(
            x
        )

        h = F.silu(
            h
        )

        h = self.conv1(
            h
        )

        time_emb = self.time_mlp(
            t
        )

        scale, shift = (
            time_emb.chunk(
                2,
                dim=1
            )
        )

        scale = scale[
            :,
            :,
            None,
            None,
            None
        ]

        shift = shift[
            :,
            :,
            None,
            None,
            None
        ]

        h = self.norm2(
            h
        )

        h = (
            h
            * (
                1.0 + scale
            )
            + shift
        )

        h = F.silu(
            h
        )

        h = self.dropout(
            h
        )

        h = self.conv2(
            h
        )

        return (
            h + residual
        )

In [11]:
class DownBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.resblock1 = ResBlock3D(
            in_channels,
            out_channels,
            time_dim
        )

        self.resblock2 = ResBlock3D(
            out_channels,
            out_channels,
            time_dim
        )

        self.downsample = nn.Conv3d(
            out_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

    def forward(
        self,
        x,
        t
    ):
        x = self.resblock1(
            x,
            t
        )

        x = self.resblock2(
            x,
            t
        )

        skip = x

        x = self.downsample(
            x
        )

        return skip, x


class UpBlock3D(nn.Module):
    def __init__(
        self,
        in_channels,
        skip_channels,
        out_channels,
        time_dim
    ):
        super().__init__()

        self.upsample = nn.ConvTranspose3d(
            in_channels,
            out_channels,
            kernel_size=4,
            stride=2,
            padding=1
        )

        self.resblock1 = ResBlock3D(
            out_channels
            + skip_channels,
            out_channels,
            time_dim
        )

        self.resblock2 = ResBlock3D(
            out_channels,
            out_channels,
            time_dim
        )

    def forward(
        self,
        x,
        skip,
        t
    ):
        x = self.upsample(
            x
        )

        if x.shape[2:] != skip.shape[2:]:
            raise ValueError(
                f"Upsample shape {x.shape} "
                f"does not match skip {skip.shape}"
            )

        x = torch.cat(
            [x, skip],
            dim=1
        )

        x = self.resblock1(
            x,
            t
        )

        x = self.resblock2(
            x,
            t
        )

        return x

In [12]:
class AttentionBlock3D(nn.Module):
    def __init__(
        self,
        channels,
        num_heads=4
    ):
        super().__init__()

        if channels % num_heads != 0:
            raise ValueError(
                "channels must be divisible "
                "by num_heads"
            )

        self.norm = nn.GroupNorm(
            num_groups=8,
            num_channels=channels
        )

        self.attention = nn.MultiheadAttention(
            embed_dim=channels,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        b, c, d, h, w = x.shape

        residual = x

        x = self.norm(
            x
        )

        # [B,C,D,H,W]
        # ->
        # [B,D*H*W,C]
        x = (
            x.permute(
                0,
                2,
                3,
                4,
                1
            )
            .reshape(
                b,
                d * h * w,
                c
            )
        )

        x, _ = self.attention(
            x,
            x,
            x,
            need_weights=False
        )

        # Restore 3D shape
        x = (
            x.reshape(
                b,
                d,
                h,
                w,
                c
            )
            .permute(
                0,
                4,
                1,
                2,
                3
            )
            .contiguous()
        )

        return x + residual


class UNet3D(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        base_channels=64,
        time_dim=256
    ):
        super().__init__()

        # Med-DDPM-style channel progression:
        # 64 -> 64 -> 128 -> 192 -> 256
        c1 = base_channels
        c2 = base_channels
        c3 = base_channels * 2
        c4 = base_channels * 3
        c5 = base_channels * 4

        self.time_embedding = nn.Sequential(
            SinusoidalTimeEmbedding(time_dim),
            nn.Linear(time_dim, time_dim),
            nn.SiLU(),
            nn.Linear(time_dim, time_dim)
        )

        # 208 x 224 x 160
        self.input_conv = nn.Conv3d(
            in_channels,
            c1,
            kernel_size=3,
            padding=1
        )

        # 208x224x160 -> 104x112x80
        self.down1 = DownBlock3D(
            c1,
            c2,
            time_dim
        )

        # 104x112x80 -> 52x56x40
        self.down2 = DownBlock3D(
            c2,
            c3,
            time_dim
        )

        # 52x56x40 -> 26x28x20
        self.down3 = DownBlock3D(
            c3,
            c4,
            time_dim
        )

        # 26x28x20 -> 13x14x10
        self.down4 = DownBlock3D(
            c4,
            c5,
            time_dim
        )

        # Bottleneck: 13 x 14 x 10
        self.mid1 = ResBlock3D(
            c5,
            c5,
            time_dim
        )

        self.mid_attention = AttentionBlock3D(
            c5,
            num_heads=4
        )

        self.mid2 = ResBlock3D(
            c5,
            c5,
            time_dim
        )

        # 13x14x10 -> 26x28x20
        self.up4 = UpBlock3D(
            in_channels=c5,
            skip_channels=c5,
            out_channels=c4,
            time_dim=time_dim
        )

        # 26x28x20 -> 52x56x40
        self.up3 = UpBlock3D(
            in_channels=c4,
            skip_channels=c4,
            out_channels=c3,
            time_dim=time_dim
        )

        # 52x56x40 -> 104x112x80
        self.up2 = UpBlock3D(
            in_channels=c3,
            skip_channels=c3,
            out_channels=c2,
            time_dim=time_dim
        )

        # 104x112x80 -> 208x224x160
        self.up1 = UpBlock3D(
            in_channels=c2,
            skip_channels=c2,
            out_channels=c1,
            time_dim=time_dim
        )

        self.output_norm = nn.GroupNorm(
            num_groups=8,
            num_channels=c1
        )

        self.output_conv = nn.Conv3d(
            c1,
            out_channels,
            kernel_size=3,
            padding=1
        )

        nn.init.zeros_(
            self.output_conv.weight
        )
        nn.init.zeros_(
            self.output_conv.bias
        )

    def forward(
        self,
        x,
        t
    ):
        t = self.time_embedding(t)
        x = self.input_conv(x)

        skip1, x = self.down1(x, t)
        skip2, x = self.down2(x, t)
        skip3, x = self.down3(x, t)
        skip4, x = self.down4(x, t)

        x = self.mid1(x, t)
        x = self.mid_attention(x)
        x = self.mid2(x, t)

        x = self.up4(x, skip4, t)
        x = self.up3(x, skip3, t)
        x = self.up2(x, skip2, t)
        x = self.up1(x, skip1, t)

        x = self.output_norm(x)
        x = F.silu(x)
        x = self.output_conv(x)

        return x


In [13]:

# ============================================================
# DDPM V8
# Forward diffusion for epsilon / noise prediction
# ============================================================

def q_sample(
    x0,
    t,
    noise=None
):
    if noise is None:
        noise = torch.randn_like(
            x0
        )

    device = x0.device

    sqrt_alpha = (
        sqrt_alphas_cumprod
        .to(device)[t]
        .view(
            -1,
            1,
            1,
            1,
            1
        )
    )

    sqrt_one_minus_alpha = (
        sqrt_one_minus_alphas_cumprod
        .to(device)[t]
        .view(
            -1,
            1,
            1,
            1,
            1
        )
    )

    xt = (
        sqrt_alpha
        * x0
        +
        sqrt_one_minus_alpha
        * noise
    )

    return xt, noise


In [ ]:

# ============================================================
# DDPM V8
# EMA + epsilon/noise-prediction training
# ============================================================

class EMA:
    def __init__(
        self,
        model,
        decay=0.9999
    ):
        self.decay = decay

        self.ema_model = copy.deepcopy(
            model
        )

        self.ema_model.eval()

        for parameter in (
            self.ema_model.parameters()
        ):
            parameter.requires_grad = False


    @torch.no_grad()
    def update(
        self,
        model
    ):
        ema_parameters = dict(
            self.ema_model.named_parameters()
        )

        model_parameters = dict(
            model.named_parameters()
        )

        for name, parameter in (
            model_parameters.items()
        ):
            ema_parameters[name].mul_(
                self.decay
            ).add_(
                parameter,
                alpha=(
                    1.0
                    - self.decay
                )
            )

        ema_buffers = dict(
            self.ema_model.named_buffers()
        )

        model_buffers = dict(
            model.named_buffers()
        )

        for name, buffer in (
            model_buffers.items()
        ):
            ema_buffers[name].copy_(
                buffer
            )


def train_ddpm(
    model,
    ema,
    train_loader,
    epochs,
    optimizer,
    device,
    checkpoint_dir="ddpm_v8_checkpoints"
):
    os.makedirs(
        checkpoint_dir,
        exist_ok=True
    )

    loss_history_path = os.path.join(
        checkpoint_dir,
        "ddpm_v8_loss_history.npy"
    )

    # V8 always trains from scratch.
    # No previous checkpoint is loaded.
    loss_history = []

    use_amp = (
        device.type == "cuda"
    )

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(
        epochs
    ):
        model.train()
        epoch_loss = 0.0

        for batch_idx, batch in enumerate(
            train_loader
        ):
            x0 = batch["image"].to(
                device,
                non_blocking=True
            )

            t = torch.randint(
                low=0,
                high=timesteps,
                size=(x0.shape[0],),
                device=device,
                dtype=torch.long
            )

            noise = torch.randn_like(
                x0
            )

            xt, noise = q_sample(
                x0=x0,
                t=t,
                noise=noise
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):
                predicted_noise = model(
                    xt,
                    t
                )

                loss = F.l1_loss(
                    predicted_noise,
                    noise
                )

            scaler.scale(
                loss
            ).backward()

            scaler.unscale_(
                optimizer
            )

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            scaler.step(
                optimizer
            )
            scaler.update()

            ema.update(
                model
            )

            epoch_loss += loss.item()

            if (
                batch_idx + 1
            ) % 10 == 0:
                print(
                    f"Epoch {epoch + 1}/{epochs} | "
                    f"Batch {batch_idx + 1}/{len(train_loader)} | "
                    f"L1={loss.item():.6f}"
                )

        avg_loss = (
            epoch_loss
            / len(train_loader)
        )

        loss_history.append(
            avg_loss
        )

        print(
            f"Epoch {epoch + 1} completed | "
            f"L1={avg_loss:.6f}"
        )

        checkpoint_path = os.path.join(
            checkpoint_dir,
            f"ddpm_v8_epoch_{epoch + 1:03d}.pt"
        )

        save_checkpoint(
            model=model,
            ema=ema,
            optimizer=optimizer,
            epoch=epoch + 1,
            path=checkpoint_path
        )

        np.save(
            loss_history_path,
            np.asarray(
                loss_history,
                dtype=np.float32
            )
        )

        print(
            "Saved:",
            checkpoint_path
        )

    return np.asarray(
        loss_history,
        dtype=np.float32
    )


In [ ]:
def save_checkpoint(
    model,
    ema,
    optimizer,
    epoch,
    path
):
    torch.save(
        {
            "epoch":
                epoch,

            "model_state_dict":
                model.state_dict(),

            "ema_state_dict":
                ema.ema_model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict()
        },
        path
    )




In [ ]:

# ============================================================
# DDPM V8
# Epsilon-prediction ancestral sampling
#
# Snapshots are indexed by COMPLETED reverse steps:
#   0   = initial Gaussian noise
#   50  = after 50 reverse denoising steps
#   100 = after 100 reverse denoising steps
#   150 = after 150 reverse denoising steps
#   200 = after 200 reverse denoising steps
#   250 = final generated MRI
# ============================================================

@torch.no_grad()
def sample_ddpm_with_progress(
    model,
    shape,
    device,
    capture_steps=(
        0,
        50,
        100,
        150,
        200,
        250
    )
):
    model.eval()

    x = torch.randn(
        shape,
        device=device
    )

    snapshots = {}

    if 0 in capture_steps:
        snapshots[0] = (
            x.detach()
            .cpu()
            .clone()
        )


    sqrt_alpha_bar = (
        sqrt_alphas_cumprod
        .to(device)
    )

    sqrt_one_minus_alpha_bar = (
        sqrt_one_minus_alphas_cumprod
        .to(device)
    )

    coef1 = (
        posterior_mean_coef1
        .to(device)
    )

    coef2 = (
        posterior_mean_coef2
        .to(device)
    )

    posterior_var = (
        posterior_variance
        .to(device)
    )


    completed_steps = 0


    for t in reversed(
        range(timesteps)
    ):
        t_batch = torch.full(
            (
                shape[0],
            ),
            t,
            device=device,
            dtype=torch.long
        )


        # Model predicts epsilon/noise directly
        predicted_noise = model(
            x,
            t_batch
        )


        # Recover x0 from epsilon prediction
        alpha_t = (
            sqrt_alpha_bar[t]
        )

        sigma_t = (
            sqrt_one_minus_alpha_bar[t]
        )

        x0_pred = (
            x
            -
            sigma_t
            * predicted_noise
        ) / alpha_t


        x0_pred = torch.clamp(
            x0_pred,
            -1.0,
            1.0
        )


        model_mean = (
            coef1[t]
            * x0_pred
            +
            coef2[t]
            * x
        )


        if t > 0:
            noise = torch.randn_like(
                x
            )

            x = (
                model_mean
                +
                torch.sqrt(
                    posterior_var[t]
                )
                * noise
            )
        else:
            x = model_mean


        completed_steps += 1


        if completed_steps in capture_steps:
            snapshots[
                completed_steps
            ] = (
                x.detach()
                .cpu()
                .clone()
            )


        if (
            completed_steps % 50
            == 0
        ):
            print(
                "Completed reverse steps:",
                completed_steps,
                "/",
                timesteps
            )


    x = torch.clamp(
        x,
        -1.0,
        1.0
    )


    snapshots[timesteps] = (
        x.detach()
        .cpu()
        .clone()
    )


    return x, snapshots


In [ ]:

# ============================================================
# DDPM V8
# Model initialisation
#
# base_channels = 64
# channel progression:
# 64 -> 64 -> 128 -> 192 -> 256
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(
    "Device:",
    device
)


model = UNet3D(
    in_channels=1,
    out_channels=1,
    base_channels=64,
    time_dim=256
).to(device)


ema = EMA(
    model,
    decay=0.9999
)


optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-5,
    weight_decay=1e-4
)


total_parameters = sum(
    p.numel()
    for p in model.parameters()
)

trainable_parameters = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print()
print("=" * 70)
print("DDPM V8 SETTINGS")
print("=" * 70)

print(
    "Input resolution:",
    "208 x 224 x 160"
)

print(
    "Diffusion timesteps:",
    timesteps
)

print(
    "Base channels:",
    64
)

print(
    "Channel progression:",
    "64 -> 64 -> 128 -> 192 -> 256"
)

print(
    "Total parameters:",
    f"{total_parameters:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "Prediction target:",
    "epsilon / noise prediction"
)

print(
    "Training epochs:",
    100
)



In [ ]:

# ============================================================
# DDPM V8
# Full GPU smoke test
#
# Checks:
# - input shape
# - 250 timesteps
# - base_channels = 64
# - epsilon prediction output shape
# - finite loss
# - backward pass
# - finite gradients
# - peak GPU memory
# ============================================================

import gc


print("=" * 72)
print("DDPM V8 — BASE64 EPSILON SMOKE TEST")
print("=" * 72)


gc.collect()

if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


batch = next(
    iter(train_loader)
)


x0 = (
    batch["image"]
    [:1]
    .to(device)
)


print(
    "Input:",
    tuple(x0.shape)
)

print(
    "Diffusion timesteps:",
    timesteps
)

print(
    "Expected channels:",
    "64 -> 64 -> 128 -> 192 -> 256"
)


assert tuple(
    x0.shape
) == (
    1,
    1,
    208,
    224,
    160
)

assert timesteps == 250


t = torch.randint(
    low=0,
    high=timesteps,
    size=(1,),
    device=device,
    dtype=torch.long
)


noise = torch.randn_like(
    x0
)


xt, noise = q_sample(
    x0=x0,
    t=t,
    noise=noise
)


optimizer.zero_grad(
    set_to_none=True
)


use_amp = (
    device.type == "cuda"
)


with torch.cuda.amp.autocast(
    enabled=use_amp
):
    predicted_noise = model(
        xt,
        t
    )


    assert (
        predicted_noise.shape
        ==
        noise.shape
    )


    loss = F.l1_loss(
        predicted_noise,
        noise
    )



print(
    "Predicted noise:",
    tuple(
        predicted_noise.shape
    )
)

print(
    "Total loss:",
    loss.item()
)

assert torch.isfinite(
    loss
)

print(
    "Finite loss: PASS"
)


loss.backward()


gradient_found = False
gradient_finite = True


for name, parameter in (
    model.named_parameters()
):
    if parameter.grad is not None:
        gradient_found = True

        if not torch.isfinite(
            parameter.grad
        ).all():
            gradient_finite = False
            print(
                "Non-finite gradient:",
                name
            )
            break


assert gradient_found
assert gradient_finite


print(
    "Backward: PASS"
)

print(
    "Finite gradients: PASS"
)


if device.type == "cuda":
    allocated = (
        torch.cuda.memory_allocated()
        /
        1024**3
    )

    reserved = (
        torch.cuda.memory_reserved()
        /
        1024**3
    )

    peak = (
        torch.cuda.max_memory_allocated()
        /
        1024**3
    )

    print()
    print(
        "GPU memory:"
    )

    print(
        f"Allocated: {allocated:.2f} GB"
    )

    print(
        f"Reserved:  {reserved:.2f} GB"
    )

    print(
        f"Peak:      {peak:.2f} GB"
    )


print()
print("=" * 72)
print("FINAL RESULT")
print("=" * 72)

print(
    "250 diffusion steps: PASS"
)

print(
    "Base channels 64: PASS"
)

print(
    "Epsilon prediction: PASS"
)

print(
    "Forward: PASS"
)

print(
    "Backward: PASS"
)

print(
    "DDPM V8 is ready for training."
)


In [ ]:

# ============================================================
# DDPM V8
# Formal training: 100 epochs
# ============================================================

loss_history = train_ddpm(
    model=model,
    ema=ema,
    train_loader=train_loader,
    epochs=100,
    optimizer=optimizer,
    device=device,
    checkpoint_dir=(
        "ddpm_v8_checkpoints"
    ),)


In [ ]:

# ============================================================
# DDPM V8
# Training loss history
# ============================================================

loss_history = np.load(
    "ddpm_v8_checkpoints/"
    "ddpm_v8_loss_history.npy"
)


print(
    "Number of epochs:",
    len(loss_history)
)


total_loss = (
    loss_history[:, 0]
)

global_loss = (
    loss_history[:, 1]
)

foreground_loss = (
    loss_history[:, 2]
)


for i in range(
    len(loss_history)
):
    print(
        f"Epoch {i + 1:03d} | "
        f"Total="
        f"{total_loss[i]:.6f} | "
        f"Global="
        f"{global_loss[i]:.6f} | "
        f"FG="
        f"{foreground_loss[i]:.6f}"
    )


plt.figure(
    figsize=(9, 5)
)

plt.plot(
    np.arange(
        1,
        len(loss_history) + 1
    ),
    total_loss,
    label="Total"
)

plt.plot(
    np.arange(
        1,
        len(loss_history) + 1
    ),
    global_loss,
    label="Global"
)

plt.plot(
    np.arange(
        1,
        len(loss_history) + 1
    ),
    foreground_loss,
    label="Foreground"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "epsilon-prediction MSE"
)

plt.title(
    "DDPM V8 Training Loss"
)

plt.legend()

plt.grid(
    alpha=0.3
)

plt.show()


In [ ]:

# ============================================================
# DDPM V8
# Generate final MRI and save denoising progression
# ============================================================

torch.manual_seed(
    42
)


generated_v8, diffusion_progress = (
    sample_ddpm_with_progress(
        model=ema.ema_model,
        shape=(
            1,
            1,
            208,
            224,
            160
        ),
        device=device,
        capture_steps=(
            0,
            50,
            100,
            150,
            200,
            250
        )
    )
)


print(
    "Generated shape:",
    generated_v8.shape
)

print(
    "Generated min/max:",
    generated_v8.min().item(),
    generated_v8.max().item()
)

print(
    "Captured reverse steps:",
    sorted(
        diffusion_progress.keys()
    )
)


In [ ]:

# ============================================================
# DDPM V8
# Denoising progression:
# 0, 50, 100, 150, 200, 250 reverse steps
# ============================================================

progress_steps = [
    0,
    50,
    100,
    150,
    200,
    250
]


z_slice = 80


plt.figure(
    figsize=(18, 4)
)


for i, step in enumerate(
    progress_steps
):
    volume = (
        diffusion_progress[
            step
        ][
            0,
            0
        ]
        .numpy()
    )


    display_volume = np.clip(
        (
            volume + 1.0
        )
        / 2.0,
        0.0,
        1.0
    )


    plt.subplot(
        1,
        6,
        i + 1
    )


    plt.imshow(
        display_volume[
            :,
            :,
            z_slice
        ],
        cmap="gray",
        vmin=0,
        vmax=1
    )


    plt.title(
        f"Step {step}"
    )


    plt.axis(
        "off"
    )


plt.suptitle(
    "DDPM V8 Reverse-Diffusion Progression"
)

plt.tight_layout()

plt.show()


# Keep final volume variables for the following cells
gen = (
    generated_v8[
        0,
        0
    ]
    .detach()
    .cpu()
    .numpy()
)


gen_display = np.clip(
    (
        gen + 1.0
    )
    / 2.0,
    0.0,
    1.0
)


In [ ]:
print(
    "Raw [-1,1] statistics"
)

print(
    "Min:",
    gen.min()
)

print(
    "Max:",
    gen.max()
)

print(
    "Mean:",
    gen.mean()
)

print(
    "Std:",
    gen.std()
)

print()

print(
    "Display [0,1] statistics"
)

print(
    "Min:",
    gen_display.min()
)

print(
    "Max:",
    gen_display.max()
)

print(
    "Mean:",
    gen_display.mean()
)

print(
    "Std:",
    gen_display.std()
)

print()

print(
    "P1:",
    np.percentile(
        gen_display,
        1
    )
)

print(
    "P50:",
    np.percentile(
        gen_display,
        50
    )
)

print(
    "P99:",
    np.percentile(
        gen_display,
        99
    )
)


In [ ]:
volume = gen_display

x_mid = (
    volume.shape[0] // 2
)

y_mid = (
    volume.shape[1] // 2
)

z_mid = (
    volume.shape[2] // 2
)

plt.figure(
    figsize=(12, 4)
)

plt.subplot(
    1,
    3,
    1
)

plt.imshow(
    volume[
        x_mid,
        :,
        :
    ].T,
    cmap="gray",
    origin="lower",
    vmin=0,
    vmax=1
)

plt.title(
    "Sagittal"
)

plt.axis(
    "off"
)

plt.subplot(
    1,
    3,
    2
)

plt.imshow(
    volume[
        :,
        y_mid,
        :
    ].T,
    cmap="gray",
    origin="lower",
    vmin=0,
    vmax=1
)

plt.title(
    "Coronal"
)

plt.axis(
    "off"
)

plt.subplot(
    1,
    3,
    3
)

plt.imshow(
    volume[
        :,
        :,
        z_mid
    ],
    cmap="gray",
    vmin=0,
    vmax=1
)

plt.title(
    "Axial"
)

plt.axis(
    "off"
)

plt.tight_layout()

plt.show()
